# Segment correlation distance experiment

This notebook computes the staff-approved Pearson-distance transform

$D_{ij} = \sqrt{2 \times (1 - r_{ij})}$

The expected repo artifact is `data/docs/segment_monthly_series.parquet`, but this checkout does not contain it. The real source in this session is the downloaded parquet under the user profile, so the notebook checks both paths and uses the real data that exists.

In [1]:
from pathlib import Path
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "backend").exists() and (candidate / "notebooks").exists():
        repo_root = candidate
        break
repo_path = repo_root / "data" / "docs" / "segment_monthly_series.parquet"
download_path = Path.home() / "Downloads" / "segment_monthly_series.parquet"
series_path = next((p for p in [repo_path, download_path] if p.exists()), None)
print("repo_path_exists", repo_path.exists())
print("download_path_exists", download_path.exists())
print("repo_root", repo_root)
print("series_path", series_path)
if series_path is None:
    raise FileNotFoundError("No segment_monthly_series.parquet found in repo or Downloads")


repo_path_exists False
download_path_exists True
repo_root C:\Users\kekum\OneDrive\Desktop\ArthaDhruva\ArthaDhruva
series_path C:\Users\kekum\Downloads\segment_monthly_series.parquet


In [2]:
import pandas as pd
import numpy as np
series = pd.read_parquet(series_path)
print(series.head().to_string(index=False))
print("shape", series.shape)
print("columns", list(series.columns))
print("states", sorted(series["property_state"].unique().tolist()))
print("months", series["monthly_reporting_period"].nunique())
print("min_month", series["monthly_reporting_period"].min())
print("max_month", series["monthly_reporting_period"].max())


property_state monthly_reporting_period  n_active  n_delinquent  delinquency_rate
            CA               2020-01-01        66             0          0.000000
            CA               2020-02-01     16311             1          0.000061
            CA               2020-03-01     40321           121          0.003001
            CA               2020-04-01     75916          1298          0.017098
            CA               2020-05-01    112157          2839          0.025313
shape (1035, 5)
columns ['property_state', 'monthly_reporting_period', 'n_active', 'n_delinquent', 'delinquency_rate']
states ['CA', 'FL', 'GA', 'IL', 'IN', 'MA', 'MD', 'MI', 'MN', 'NC', 'NY', 'OH', 'PA', 'TN', 'TX']
months 69
min_month 2020-01-01
max_month 2025-09-01


In [3]:
wide = series.pivot_table(index="monthly_reporting_period", columns="property_state", values="delinquency_rate", aggfunc="mean").sort_index()
corr = wide.corr(method="pearson")
corr_array = corr.to_numpy(copy=True)
distance_array = np.sqrt(np.clip(2 * (1 - corr_array), 0, None))
distance_df = pd.DataFrame(distance_array, index=corr.index, columns=corr.columns)
for idx in range(len(distance_df.columns)):
    distance_df.iat[idx, idx] = 0.0
print("corr_shape", corr.shape)
print("distance_shape", distance_df.shape)
display(corr.head())
display(distance_df.head())


corr_shape (15, 15)
distance_shape (15, 15)


property_state,CA,FL,GA,IL,IN,MA,MD,MI,MN,NC,NY,OH,PA,TN,TX
property_state,,,,,,,,,,,,,,,
CA,1.000000,0.910555,0.871780,0.810579,0.543089,0.904235,0.962642,0.702226,0.723680,0.827819,0.964231,0.688612,0.833063,0.776652,0.896349
FL,0.910555,1.000000,0.961867,0.923205,0.729828,0.944426,0.950243,0.848064,0.863516,0.956430,0.833555,0.843665,0.930087,0.888103,0.959513
GA,0.871780,0.961867,1.000000,0.976973,0.849848,0.967824,0.945641,0.927753,0.939035,0.985629,0.753790,0.930968,0.973203,0.968637,0.988810
IL,0.810579,0.923205,0.976973,1.000000,0.901428,0.962644,0.914390,0.966674,0.964549,0.975182,0.680214,0.967348,0.982675,0.979178,0.968395
IN,0.543089,0.729828,0.849848,0.901428,1.000000,0.807107,0.707510,0.960904,0.937949,0.858143,0.367133,0.965826,0.892073,0.931981,0.812316


property_state,CA,FL,GA,IL,IN,MA,MD,MI,MN,NC,NY,OH,PA,TN,TX
property_state,,,,,,,,,,,,,,,
CA,0.000000,0.422954,0.506398,0.615502,0.955940,0.437642,0.273344,0.771718,0.743397,0.586824,0.267464,0.789162,0.577818,0.668353,0.455303
FL,0.422954,0.000000,0.276164,0.391905,0.735080,0.333389,0.315459,0.551247,0.522463,0.295196,0.576966,0.559170,0.373934,0.473068,0.284560
GA,0.506398,0.276164,0.000000,0.214602,0.547999,0.253678,0.329724,0.380124,0.349186,0.169535,0.701726,0.371569,0.231504,0.250452,0.149601
IL,0.615502,0.391905,0.214602,0.000000,0.444010,0.273336,0.413788,0.258169,0.266274,0.222789,0.799732,0.255548,0.186143,0.204071,0.251415
IN,0.955940,0.735080,0.547999,0.444010,0.000000,0.621116,0.764840,0.279627,0.352280,0.532648,1.125048,0.261433,0.464601,0.368833,0.612672


## Validate the matrix properties

Check symmetry, diagonal agreement, and bounds.

In [4]:
cutoff = np.sqrt(2 * (1 - 0.95))
validation = {
    "symmetry_ok": bool(np.allclose(corr.to_numpy(), corr.to_numpy().T, atol=1e-12)),
    "diag_corr_ok": bool(np.allclose(np.diag(corr.to_numpy()), 1.0, atol=1e-12)),
    "diag_distance_ok": bool(np.allclose(np.diag(distance_df.to_numpy()), 0.0, atol=1e-12)),
    "distance_nonnegative": bool((distance_df.to_numpy()[~np.eye(distance_df.shape[0], dtype=bool)] >= 0).all()),
    "pearson_offdiagonal_min": float(corr.to_numpy()[~np.eye(corr.shape[0], dtype=bool)].min()),
    "pearson_offdiagonal_max": float(corr.to_numpy()[~np.eye(corr.shape[0], dtype=bool)].max()),
    "distance_offdiagonal_min": float(distance_df.to_numpy()[~np.eye(distance_df.shape[0], dtype=bool)].min()),
    "distance_offdiagonal_max": float(distance_df.to_numpy()[~np.eye(distance_df.shape[0], dtype=bool)].max()),
    "cutoff_d": float(cutoff),
}
print(validation)


{'symmetry_ok': True, 'diag_corr_ok': True, 'diag_distance_ok': True, 'distance_nonnegative': True, 'pearson_offdiagonal_min': 0.36713326267020313, 'pearson_offdiagonal_max': 0.9914915430046535, 'distance_offdiagonal_min': 0.13044889417198235, 'distance_offdiagonal_max': 1.125048209926843, 'cutoff_d': 0.31622776601683805}


## Pairwise examples and threshold equivalence

The threshold `r >= 0.95` is equivalent to `D <= sqrt(2 * (1 - 0.95))`.

In [5]:
rows = []
for i, state_i in enumerate(corr.columns):
    for state_j in corr.columns[i + 1:]:
        r = float(corr.loc[state_i, state_j])
        d = float(np.sqrt(2 * (1 - r)))
        rows.append({
            "state_i": state_i,
            "state_j": state_j,
            "pearson_r": r,
            "distance_d": d,
            "r_ge_0_95": r >= 0.95,
            "d_le_cutoff": d <= cutoff,
        })
pair_df = pd.DataFrame(rows)
display(pair_df.sort_values("pearson_r", ascending=False).head(5))
print("threshold_match", bool(pair_df["r_ge_0_95"].equals(pair_df["d_le_cutoff"])))


,state_i,state_j,pearson_r,distance_d,r_ge_0_95,d_le_cutoff
80,MI,OH,0.991492,0.130449,True,True
38,GA,TX,0.988810,0.149601,True,True
33,GA,NC,0.985629,0.169535,True,True
47,IL,PA,0.982675,0.186143,True,True
48,IL,TN,0.979178,0.204071,True,True


threshold_match True


## Compare to the existing graph export

The original graph keeps all state pairs with `r >= 0.95`; the distance cutoff should select the same pairs.

In [6]:
import json
graph_path = repo_root / "backend" / "risk-engine" / "src" / "main" / "resources" / "segment_correlation_graph.json"
with graph_path.open("r", encoding="utf-8") as f:
    graph = json.load(f)
existing_pairs = {(min(e["source"], e["target"]), max(e["source"], e["target"])) for e in graph["edges"]}
threshold_pairs = {(min(r.state_i, r.state_j), max(r.state_i, r.state_j)) for _, r in pair_df.iterrows() if r["r_ge_0_95"]}
print("graph_edge_count", len(graph["edges"]))
print("threshold_edge_count", len(threshold_pairs))
print("graph_matches_threshold", existing_pairs == threshold_pairs)


graph_edge_count 41
threshold_edge_count 41
graph_matches_threshold True
